# DreamDesk test 1: train world-model priors on 7, 15 and 23 apps

As pre-registered in `experiments/scale_prereg.md`. No real app runs here; the laptop fine-tunes and scores afterwards.

**How to run:**
1. Add the uploaded `dreamdesk_scale.zip` as input (Add Input → Your Datasets).
2. Settings: Accelerator **GPU T4 x1**. Internet can stay off.
3. Run cells 1–3 by hand once. Cell 3 says whether GPU or CPU is faster and how long it will take.
4. Then **Save Version → Save & Run All (Commit)**. It runs in the background for up to 12 h.
5. If it stops early: make a new version with the previous version's **output** added as an input. Finished priors are kept and unfinished ones resume after their last epoch.
6. Download `scale_results.zip` from the output.

In [ ]:
# 1. Configuration, fixed by experiments/scale_prereg.md
PRIORS = ["prior7", "prior15", "prior23"]
SEEDS = [0, 1, 2]
EPOCHS = 8
PARALLEL = 3      # trainings at once (the GPU is shared; Kaggle has 4 CPU cores)
THREADS = 1

In [ ]:
# 2. Setup: find the input (unzipping if needed), copy it somewhere writable, restore earlier output.
import glob, os, shutil, subprocess, sys, json, time, zipfile
def walk(top):
    for d, dirs, files in os.walk(top, followlinks=True):
        yield d, files
src = [d for d, files in walk("/kaggle/input") if "world_model.py" in files]
if not src:
    zips = [os.path.join(d, f) for d, files in walk("/kaggle/input") for f in files if f.endswith(".zip") and "scale" in f]
    assert zips, "no dataset attached: Add Input -> Your Datasets -> dreamdesk-scale"
    zipfile.ZipFile(zips[0]).extractall("/kaggle/tmp_unzip")
    src = [d for d, files in walk("/kaggle/tmp_unzip") if "world_model.py" in files]
root = os.path.dirname(src[0])
W = "/kaggle/working/dd"
if not os.path.exists(W):
    shutil.copytree(root, W)
os.chdir(W)
os.makedirs("runs/scale", exist_ok=True)
for d, files in walk("/kaggle/input"):   # an earlier version's output: resume
    if "/runs/scale/" in d:
        for f in files:
            dst = f"runs/scale/{os.path.basename(d)}/{f}"
            if not os.path.exists(dst):
                os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(os.path.join(d, f), dst)
for p in PRIORS:
    print(p, len(glob.glob(f"data/{p}/*.pkl")), "apps")
import torch; print("cuda:", torch.cuda.is_available())

In [ ]:
# 3. Speed check: 40 batches on GPU and on CPU, then the estimate.
def bench(device):
    out = f"/tmp/bench_{device}"; shutil.rmtree(out, ignore_errors=True)
    t = time.time()
    r = subprocess.run([sys.executable, "-W", "ignore", "-m", "dreamdesk.world_model", "train", "--data", "data/prior23", "--out", out,
                        "--seed", "0", "--epochs", "1", "--limit", "40", "--threads", str(THREADS), "--device", device],
                       capture_output=True, text=True)
    assert r.returncode == 0, r.stdout[-2000:] + r.stderr[-3000:]
    row = json.loads(open(f"{out}/train_seed0.log").readline())
    return row["seconds"] / 40, time.time() - t
gpu, gt = bench("cuda"); cpu, ct = bench("cpu")
DEVICE = "cuda" if gpu < cpu else "cpu"
print(f"per batch: cuda {gpu*1000:.0f} ms, cpu {cpu*1000:.0f} ms -> using {DEVICE}")
batches = {p: None for p in PRIORS}
r = subprocess.run([sys.executable, "-c", "import sys; sys.argv=['x']; from dreamdesk.world_model import *; from dreamdesk.wm_data import load; from pathlib import Path; import json;"
                    "print(json.dumps({p: sum(len(token_batches(Prepared(load(f), {}), load(f).transitions['train'])) for f in sorted(Path('data/'+p).glob('*.pkl'))) for p in %r}))" % PRIORS],
                   capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-1500:])

In [ ]:
# 4. Train every prior x seed. Finished ones are skipped; unfinished ones resume after their last epoch.
from concurrent.futures import ThreadPoolExecutor
import threading
DEVICE = globals().get("DEVICE", "cuda")
def done(p, s):
    log = f"runs/scale/{p}/train_seed{s}.log"
    return os.path.exists(log) and any(json.loads(l)["epoch"] == EPOCHS - 1 for l in open(log))
def train(job):
    p, s = job
    if done(p, s):
        return f"{p} seed {s}: already done"
    with open(f"runs/scale/{p}_seed{s}_stdout.txt", "a") as f:
        r = subprocess.run([sys.executable, "-u", "-W", "ignore", "-m", "dreamdesk.world_model", "train", "--data", f"data/{p}",
                            "--out", f"runs/scale/{p}", "--seed", str(s), "--epochs", str(EPOCHS), "--threads", str(THREADS),
                            "--device", DEVICE, "--resume"], stdout=f, stderr=subprocess.STDOUT)
    return f"{p} seed {s}: exit {r.returncode}"
jobs = [(p, s) for s in SEEDS for p in PRIORS]
def progress():
    while True:
        time.sleep(1800)
        for p, s in jobs:
            log = f"runs/scale/{p}/train_seed{s}.log"
            n = sum(1 for _ in open(log)) if os.path.exists(log) else 0
            print(time.strftime("%H:%M"), f"{p} seed {s}: {n}/{EPOCHS} epochs", flush=True)
threading.Thread(target=progress, daemon=True).start()
with ThreadPoolExecutor(PARALLEL) as ex:
    for msg in ex.map(train, jobs):
        print(time.strftime("%H:%M"), msg, flush=True)

In [ ]:
# 5. Summary and the archive to download.
for p in PRIORS:
    for s in SEEDS:
        log = f"runs/scale/{p}/train_seed{s}.log"
        rows = [json.loads(l) for l in open(log)] if os.path.exists(log) else []
        print(p, s, "DONE" if done(p, s) else "NOT DONE", rows[-1] if rows else "")
shutil.make_archive("/kaggle/working/scale_results", "zip", W, "runs/scale")
print("saved /kaggle/working/scale_results.zip")